# Full-test spatial-frequency follow-up

Use the updated project source and the saved full EmoSet Kaggle workspace as input datasets. This evaluates six frozen checkpoints on 17,715 test images under 16 fixed conditions. No training is performed. Results are exploratory because this test set has already been inspected. One CUDA GPU is sufficient. Enable Internet only if the optional Markdown package must be installed.


In [ ]:
from pathlib import Path
import os,sys,subprocess,shutil,importlib.util
import torch
assert torch.cuda.is_available(), 'Select a GPU accelerator'
roots=[p.parents[2] for p in Path('/kaggle/input').rglob('src/ecc/spatial_frequency.py')]
roots=list(dict.fromkeys(roots))
assert len(roots)==1, f'Attach exactly one UPDATED project source with src/ecc/spatial_frequency.py; found {roots}'
REPO=Path('/kaggle/working/sp_frequency_source')
shutil.copytree(roots[0],REPO,dirs_exist_ok=True)
inputs=[p.parents[2] for p in Path('/kaggle/input').rglob('data/emoset_full/test.csv')]
inputs=[p for p in inputs if (p/'emoset_graph_alpha_sweep_2gpu/runs/graph_a0p3_seed2/best.pt').exists()]
inputs=list(dict.fromkeys(inputs))
assert len(inputs)==1, f'Attach the extracted previous workspace, including test JPEGs and all six best.pt files; found {inputs}'
SOURCE=inputs[0]
OUTPUT=Path('/kaggle/working/full_spatial_frequency')
if importlib.util.find_spec('markdown') is None:
    subprocess.run([sys.executable,'-m','pip','install','markdown'],check=True)
env=os.environ.copy();env['PYTHONPATH']=str(REPO/'src')
env['MPLCONFIGDIR']='/kaggle/working/matplotlib_cache'
print('Source code:',REPO,'Saved workspace:',SOURCE,'GPU:',torch.cuda.get_device_name(0))


In [ ]:
subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_spatial_frequency.py'),
                '--root',str(SOURCE),'--out',str(OUTPUT),'--per-class','0','--batch-size','64'],
               cwd=REPO,env=env,check=True)
# The report module uses outputs/full_spatial_frequency relative to its cwd.
REPORT_ROOT=Path('/kaggle/working/report_workspace')
(REPORT_ROOT/'outputs').mkdir(parents=True,exist_ok=True)
ALIAS=REPORT_ROOT/'outputs/full_spatial_frequency'
if not ALIAS.exists(): ALIAS.symlink_to(OUTPUT,target_is_directory=True)
subprocess.run([sys.executable,str(REPO/'scripts/report_spatial_frequency.py')],
               cwd=REPORT_ROOT,env=env,check=True)


In [ ]:
import zipfile
ARCHIVE=Path('/kaggle/working/spatial_frequency_results.zip')
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for f in OUTPUT.rglob('*'):
        if f.is_file() and f.suffix!='.npy':z.write(f,f.relative_to(OUTPUT))
print('Report:',OUTPUT/'REPORT.html')
print('Download:',ARCHIVE,'MiB:',round(ARCHIVE.stat().st_size/1024**2,1))
